In [ ]:
import pandas as pd
import itertools
import statsmodels.api as sm

def exhaustive_regression(df, phenotype, features, covariates=None):
    """
    Perform exhaustive pairwise regression with interaction terms.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing phenotype, features, and covariates.
    phenotype : str
        Name of the phenotype column.
    features : list[str]
        List of feature (e.g. protein) column names.
    covariates : list[str], optional
        List of covariate column names.

    Returns
    -------
    results : pd.DataFrame
        DataFrame with feature1, feature2, interaction_pval, and coefficients.
    """
    y = df[phenotype]
    covariates = covariates or []
    results = []

    for f1, f2 in itertools.combinations(features, 2):
        X = df[[f1, f2] + covariates].copy()
        X["interaction"] = df[f1] * df[f2]
        X = sm.add_constant(X)

        model = sm.OLS(y, X).fit()
        pval = model.pvalues["interaction"]
        results.append({
            "feature1": f1,
            "feature2": f2,
            "interaction_pval": pval,
            "interaction_coef": model.params["interaction"]
        })

    return pd.DataFrame(results)


In [ ]:
# Example DataFrame
df = pd.DataFrame({
    "phenotype": [1.2, 3.4, 2.5, 4.2, 3.8],
    "protA": [0.1, 0.3, 0.2, 0.5, 0.4],
    "protB": [0.7, 0.6, 0.8, 0.9, 0.7],
    "protC": [0.4, 0.2, 0.3, 0.6, 0.5],
    "age": [22, 25, 30, 40, 35],
    "sex": [0, 1, 1, 0, 0]
})

res = exhaustive_regression(df, "phenotype", ["protA", "protB", "protC"], covariates=["age", "sex"])
print(res)